# **Hybrid RAG (LlamaIndex)**

LlamaIndex 버전의 Hybrid RAG입니다. LangChain 버전(`hybrid_rag.ipynb`)과 동일한 hybrid search 방식(vector + BM25)을 사용하되, RAG 전용 라이브러리인 LlamaIndex의 컴포넌트로 구현합니다.

| LangChain | LlamaIndex | 차이점 |
|---|---|---|
| `RecursiveCharacterTextSplitter` | `SentenceSplitter` | 문자 수 기준 → **문장 경계** 기준 청킹 |
| `Chroma.from_documents()` | `VectorStoreIndex` | 일반 문서 저장소 → **RAG 인덱스** (임베딩·저장·검색 통합) |
| `BM25Retriever` (langchain) | `BM25Retriever` (llama_index) | LlamaIndex Node와 **네이티브 호환** |
| `EnsembleRetriever` | `HybridRetriever(BaseRetriever)` | **QueryBundle** 기반 인터페이스 |
| LCEL 체인 | `RetrieverQueryEngine` | 범용 체인 → **RAG 전용** 쿼리 엔진 |

Research Paper: [paper1](https://arxiv.org/pdf/2408.05141) and [paper2](https://arxiv.org/pdf/2408.04948)

## **Install Dependencies**

## **Initial Setup**

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# load embedding model
# LangChain: OpenAIEmbeddings()
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",  # 모델 변경
    # dimensions=512,                  # 임베딩 차원 축소 (3 계열에서만 지원)
    # chunk_size=500,                  # 한 번에 임베딩 요청할 텍스트 개수
    # max_retries=3,                   # 실패 시 재시도 횟수
)
Settings.embed_model = embed_model

c:\Users\김진규\rag-cookbooks\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [3]:
# load data
# CSVReader는 Windows cp949 환경에서 인코딩 충돌이 발생할 수 있어 pandas로 직접 로드
import pandas as pd
from llama_index.core import Document

df = pd.read_csv("../data/context.csv", encoding="utf-8")
documents = [Document(text=str(row.to_dict()), metadata={"row": i}) for i, row in df.iterrows()]

In [ ]:
# process documents with IngestionPipeline + SentenceSplitter
#
# LangChain: RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
#   → \n\n, \n 등 문장 단위로 자름, 문장 중간에 잘릴 수 있음 -> (문장 -> 줄 -> 단어 순서로 쪼개짐)
#
# LlamaIndex: IngestionPipeline + SentenceSplitter
#   → 문장 경계를 인식해서 자름, 청크마다 완전한 문장 보장 -> 문장 단위 보장에 강함 
#   → IngestionPipeline은 변환 단계를 파이프라인으로 관리 (캐싱, 확장 가능)
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=500, chunk_overlap=0),
    ]
)
nodes = pipeline.run(documents=documents)
print(f"Total nodes: {len(nodes)}")

Total nodes: 971


In [7]:
print(pipeline)

name='default' project_name='Default' transformations=[SentenceSplitter(include_metadata=True, include_prev_next_rel=True, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x00000262580A7B90>, id_func=<function default_id_func at 0x0000026253476C00>, chunk_size=500, chunk_overlap=0, separator=' ', paragraph_separator='\n\n\n', secondary_chunking_regex='[^,.;。？！]+[,.;。？！]?')] documents=None readers=None vector_store=None cache=IngestionCache(nodes_key='nodes', collection='llama_cache', cache=<llama_index.core.storage.kvstore.simple_kvstore.SimpleKVStore object at 0x000002622185FD50>) docstore=None docstore_strategy=<DocstoreStrategy.UPSERTS: 'upserts'> disable_cache=False


In [8]:
# create vector index
#
# LangChain: Chroma.from_documents(documents, embeddings)
#   → 일반 벡터 저장소
#
# LlamaIndex: VectorStoreIndex
#   → 임베딩 생성 / 저장 / 검색을 하나의 RAG 인덱스 단위로 관리
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

chroma_client = chromadb.EphemeralClient() # 메모리에만 저장
chroma_collection = chroma_client.create_collection("hybrid_rag")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex(
    nodes=nodes,
    vector_store=vector_store,
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## **Retrievers**

In [9]:
# create vector retriever
# LangChain: vectorstore.as_retriever()
vector_retriever = index.as_retriever(similarity_top_k=3)

### **Keyword Retriever**

In [15]:
# create BM25 keyword retriever
#
# LangChain: BM25Retriever.from_documents(documents)
#   → LangChain Document 객체 기반
#
# LlamaIndex: BM25Retriever.from_defaults(nodes=nodes)
#   → LlamaIndex Node 구조와 네이티브 호환, 인덱스와 동일한 Node 사용
from llama_index.retrievers.bm25 import BM25Retriever
import Stemmer

bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),  # 어간 추출 (running→run 등)
    language="english",                   # 불용어 처리
)

In [16]:
# test keyword retriever
results = bm25_retriever.retrieve("what bacteria grow on macconkey agar")
for r in results:
    print(r.text[:300])
    print("---")

{'context': '[\'MacConkey agar is a selective and differential culture medium for bacteria. It is designed to selectively isolate Gram-negative and enteric (normally found in the intestinal tract) bacteria and differentiate them based on lactose fermentation. Lactose fermenters turn red or pink on M
---
Presence of bile salts inhibits swarming by Proteus species.\\n\\n\\n=== Lac positive ===\\nBy utilizing the lactose available in the medium, Lac+ bacteria such as Escherichia coli, Enterobacter and Klebsiella will produce acid, which lowers the pH of the agar below 6.8 and results in the appearance
---
The absence of a paroxysmal cough or posttussive emesis, though, makes it almost half as likely.The illness usually starts with mild respiratory symptoms include mild coughing, sneezing, or a runny nose (known as the catarrhal stage). After one or two weeks, the coughing classically develops into un
---


### **Hybrid Retriever**

In [21]:
# create hybrid retriever (vector + BM25)
#
# LangChain: EnsembleRetriever(retrievers=[...], weights=[0.5, 0.5])
#   → 내부적으로 점수 정규화 후 가중치 합산
#
# LlamaIndex: BaseRetriever 상속으로 직접 구현
#   → QueryBundle 인터페이스: 쿼리 텍스트 + 임베딩을 같이 전달 가능
#   → node_id 기준 dedup (동일 방식)
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle
from typing import List

class HybridRetriever(BaseRetriever):
      def __init__(self, vector_retriever, bm25_retriever, vector_weight=0.5, bm25_weight=0.5):
          self._vector_retriever = vector_retriever
          self._bm25_retriever = bm25_retriever
          self._vector_weight = vector_weight
          self._bm25_weight = bm25_weight
          super().__init__()

      def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
          vector_nodes = self._vector_retriever.retrieve(query_bundle)
          bm25_nodes = self._bm25_retriever.retrieve(query_bundle)

          def normalize(nodes):
              if not nodes:
                  return nodes
              scores = [n.score or 0.0 for n in nodes]
              min_s, max_s = min(scores), max(scores)
              for n in nodes:
                  n.score = (n.score - min_s) / (max_s - min_s) if max_s != min_s else 1.0
              return nodes

          vector_nodes = normalize(vector_nodes)
          bm25_nodes   = normalize(bm25_nodes)

          # 가중치 적용 후 node_id 기준 점수 합산
          score_map = {}
          node_map  = {}
          for node in vector_nodes:
              nid = node.node.node_id
              score_map[nid] = score_map.get(nid, 0.0) + self._vector_weight * (node.score or 0.0)
              node_map[nid]  = node
          for node in bm25_nodes:
              nid = node.node.node_id
              score_map[nid] = score_map.get(nid, 0.0) + self._bm25_weight * (node.score or 0.0)
              node_map[nid]  = node

          # 합산 점수 높은 순 정렬
          merged = []
          for nid, score in sorted(score_map.items(), key=lambda x: x[1], reverse=True):
              node_map[nid].score = score
              merged.append(node_map[nid])

          return merged

hybrid_retriever = HybridRetriever(vector_retriever, bm25_retriever)  # 기본 0.5/0.5

In [22]:
# test hybrid retriever
results = hybrid_retriever.retrieve("what bacteria grow on macconkey agar")
for r in results:
    print(r.text[:300])
    print("---")

{'context': '[\'MacConkey agar is a selective and differential culture medium for bacteria. It is designed to selectively isolate Gram-negative and enteric (normally found in the intestinal tract) bacteria and differentiate them based on lactose fermentation. Lactose fermenters turn red or pink on M
---
Presence of bile salts inhibits swarming by Proteus species.\\n\\n\\n=== Lac positive ===\\nBy utilizing the lactose available in the medium, Lac+ bacteria such as Escherichia coli, Enterobacter and Klebsiella will produce acid, which lowers the pH of the agar below 6.8 and results in the appearance
---
"\n "testing indicates appropriate susceptibility to the drug, doxycycline may be used to treat these infections caused by Gram-negative bacteria:\\nEscherichia coli infections\\nEnterobacter aerogenes (formerly Aerobacter aerogenes) infections\\nShigella species infections\\nAcinetobacter species (
---
The absence of a paroxysmal cough or posttussive emesis, though, makes it almost half

## **RAG Query Engine**

In [26]:
# create LLM and query engine
#
# LangChain: ChatOpenAI() + LCEL 체인 (범용 파이프라인)
#
# LlamaIndex: RetrieverQueryEngine (RAG 전용 쿼리 엔진)
#   → retrieve → synthesize 라이프사이클을 내장
#   → response_mode로 응답 합성 방식 선택 가능:
#     'compact'(기본): 컨텍스트를 LLM 창에 효율적으로 패킹
#     'refine': 청크마다 순차 답변 개선
#     'tree_summarize': 트리 구조로 요약 합성
from llama_index.llms.openai import OpenAI
from llama_index.core.query_engine import RetrieverQueryEngine

Settings.llm = OpenAI("gpt-4o")

query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    # response_mode="compact",  # 기본값
    # response_mode="refine",   # 더 정교한 답변이 필요할 때
)

In [27]:
# response
response = query_engine.query("what bacteria grow on macconkey agar")
print(response)

MacConkey agar is designed to selectively isolate Gram-negative and enteric bacteria. Bacteria that can grow on MacConkey agar include lactose fermenters like Escherichia coli, Enterobacter, and Klebsiella, which produce pink colonies. Non-lactose fermenters such as Salmonella, Proteus species, Yersinia, Pseudomonas aeruginosa, and Shigella form normal-colored colonies. Some bacteria like Serratia and Citrobacter ferment lactose slowly or weakly.


## **Preparing Data for Evaluation**

In [28]:
# create dataset
questions = ["what bacteria grow on macconkey agar", "who wrote a rose is a rose is a rose"]
responses = []
contexts = []

# Inference
for query in questions:
    response = query_engine.query(query)
    responses.append(str(response))
    contexts.append([node.text for node in hybrid_retriever.retrieve(query)])

# To dict
data = {
    "query": questions,
    "response": responses,
    "context": contexts,
}

In [29]:
# create dataset
from datasets import Dataset
dataset = Dataset.from_dict(data)

In [30]:
# create dataframe
import pandas as pd
df = pd.DataFrame(dataset)
df

,query,response,context
0,what bacteria grow on macconkey agar,MacConkey agar is designed to selectively isol...,[{'context': '[\'MacConkey agar is a selective...
1,who wrote a rose is a rose is a rose,"The phrase ""A rose is a rose is a rose"" was wr...","[{'context': '[\'The sentence ""Rose is a rose ..."


In [51]:
df['context'][1][0]

'{\'context\': \'[\\\'The sentence "Rose is a rose is a rose is a rose" was written by Gertrude Stein as part of the 1913 poem "Sacred Emily", which appeared in the 1922 book Geography and Plays. In that poem, the first "Rose" is the name of a person. Stein later used variations on the sentence in other writings, and the shortened form "A rose is a rose is a rose" is among her most famous quotations, often interpreted as meaning "things are what they are", a statement of the law of identity, "A is A". \\\\nIn Stein\\\\\\\'s view, the sentence expresses the fact that simply using the name of a thing already invokes the imagery and emotions associated with it, an idea also intensively discussed in the problem of universals debate where Peter Abelard and others used the rose as an example concept. As the quotation diffused through her own writing, and the culture at large, Stein once remarked, "Now listen! I\\\\\\\'m no fool. I know that in daily life we don\\\\\\\'t go around saying \\\\

## **Evaluation with Ragas**

[Ragas](https://docs.ragas.io/)로 RAG 파이프라인을 평가합니다.

- **Faithfulness**: 답변이 검색된 컨텍스트에 근거하는지 (환각 탐지)
- **Answer Relevancy**: 답변이 질문에 얼마나 관련 있는지

In [32]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy

# ragas 0.2.x: from_dict()는 list of dicts 형태 필요
eval_data = [
    {
        "user_input": q,
        "response": r,
        "retrieved_contexts": c,
    }
    for q, r, c in zip(data["query"], data["response"], data["context"])
]
ragas_dataset = EvaluationDataset.from_dict(eval_data)

In [ ]:
eval_data[1]

{'user_input': 'who wrote a rose is a rose is a rose',
 'response': 'The phrase "A rose is a rose is a rose" was written by Gertrude Stein.',
 'retrieved_contexts': ['{\'context\': \'[\\\'The sentence "Rose is a rose is a rose is a rose" was written by Gertrude Stein as part of the 1913 poem "Sacred Emily", which appeared in the 1922 book Geography and Plays. In that poem, the first "Rose" is the name of a person. Stein later used variations on the sentence in other writings, and the shortened form "A rose is a rose is a rose" is among her most famous quotations, often interpreted as meaning "things are what they are", a statement of the law of identity, "A is A". \\\\nIn Stein\\\\\\\'s view, the sentence expresses the fact that simply using the name of a thing already invokes the imagery and emotions associated with it, an idea also intensively discussed in the problem of universals debate where Peter Abelard and others used the rose as an example concept. As the quotation diffused th

In [33]:
result = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
)
print(result)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.9265}


In [34]:
result.to_pandas()

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,what bacteria grow on macconkey agar,[{'context': '[\'MacConkey agar is a selective...,MacConkey agar is designed to selectively isol...,1.0,0.928168
1,who wrote a rose is a rose is a rose,"[{'context': '[\'The sentence ""Rose is a rose ...","The phrase ""A rose is a rose is a rose"" was wr...",1.0,0.924825


LLM이 같아도 BM25 Stemmer → 더 정확한 키워드 매칭 → 저자 정보가 있는 청크가 상위 랭크 → LLM이 더 좋은 답변 생성 → answer_relevancy 점수 상승.